# Microsoft Agent Framework — Azure OpenAI (Responses API)

В этом примере кода вы будете использовать **Microsoft Agent Framework (MAF)** для создания простого агента на базе **Azure OpenAI** с использованием **Responses API**.

> **Примечание о миграции:** Этот пример ранее использовал Semantic Kernel с GitHub Models. Он был перенесён на Microsoft Agent Framework, а GitHub Models (устаревшие, прекращение поддержки в июле 2026 года) были заменены на Azure OpenAI, который поддерживает Responses API. `OpenAIChatClient` в MAF нацелен на стабильную конечную точку Azure OpenAI `/openai/v1/` и по умолчанию использует Responses API.

Цель этого примера — продемонстрировать шаги, которые впоследствии будут применены в дополнительных примерах кода при реализации различных агентных паттернов.


In [ ]:
%pip install agent-framework agent-framework-openai azure-identity -q


## Импорт необходимых пакетов Python


In [ ]:
import os
import random

from dotenv import load_dotenv
from IPython.display import display, HTML

from agent_framework import tool
from agent_framework.openai import OpenAIChatClient
from azure.identity import AzureCliCredential


## Определение инструмента

В Microsoft Agent Framework **инструмент** — это обычная функция Python, украшенная `@tool`, которую агент может вызвать. Ниже мы определяем инструмент, который возвращает случайное место для отпуска и избегает повторения предыдущего.


In [ ]:
# A list of vacation destinations the tool can choose from.
_DESTINATIONS = [
    "Barcelona, Spain",
    "Paris, France",
    "Berlin, Germany",
    "Tokyo, Japan",
    "Sydney, Australia",
    "New York, USA",
    "Cairo, Egypt",
    "Cape Town, South Africa",
    "Rio de Janeiro, Brazil",
    "Bali, Indonesia",
]

# Track the last destination so repeated calls avoid immediate repeats.
_last_destination: str | None = None


@tool(approval_mode="never_require")
def get_random_destination() -> str:
    """Provides a random vacation destination."""
    global _last_destination
    available = _DESTINATIONS.copy()
    if _last_destination and len(available) > 1:
        available.remove(_last_destination)
    destination = random.choice(available)
    _last_destination = destination
    return destination


In [ ]:
load_dotenv()

endpoint = os.environ["AZURE_OPENAI_ENDPOINT"]
deployment = os.environ["AZURE_OPENAI_DEPLOYMENT"]

# OpenAIChatClient targets Azure OpenAI's v1 endpoint and uses the Responses API.
# Sign in with `az login` first so AzureCliCredential can authenticate.
chat_client = OpenAIChatClient(
    model=deployment,
    azure_endpoint=endpoint,
    credential=AzureCliCredential(),
)


## Создание агента

Здесь мы создаём агента с именем `TravelAgent`.

В этом примере мы используем очень простые инструкции. Не стесняйтесь изменять эти инструкции, чтобы наблюдать, как меняется поведение агента.


In [ ]:
agent = chat_client.as_agent(
    name="TravelAgent",
    instructions="You are a helpful AI Agent that can help plan vacations for customers at random destinations",
    tools=[get_random_destination],
)


## Запуск Агента

Теперь мы можем запустить агента. Мы создаём `AgentSession`, чтобы агент запоминал разговор между ходами, затем отправляем два `user_inputs`. Первый запрашивает поездку; второй говорит, что пользователю не понравилось предложение, и просит другое — агент использует историю сессии плюс инструмент `get_random_destination`, чтобы ответить.

Вы можете изменить эти сообщения, чтобы наблюдать, как агент реагирует по-разному. Ответы **потоково** выводятся токен за токеном.


In [ ]:
user_inputs = [
    "Plan me a day trip.",
    "I don't like that destination. Plan me another vacation.",
]


async def main():
    # A session keeps conversation history across turns.
    session = agent.create_session()

    for user_input in user_inputs:
        html_output = (
            f"<div style='margin-bottom:10px'>"
            f"<div style='font-weight:bold'>User:</div>"
            f"<div style='margin-left:20px'>{user_input}</div></div>"
        )

        full_response: list[str] = []
        # Stream the agent's response token-by-token. The agent will call the
        # get_random_destination tool automatically when it needs a destination.
        async for chunk in agent.run(user_input, session=session, stream=True):
            full_response.append(str(chunk))

        html_output += (
            "<div style='margin-bottom:20px'>"
            f"<div style='font-weight:bold'>TravelAgent:</div>"
            f"<div style='margin-left:20px; white-space:pre-wrap'>{''.join(full_response)}</div></div><hr>"
        )

        display(HTML(html_output))


await main()


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Отказ от ответственности**:
Этот документ был переведен с использованием сервиса машинного перевода [Co-op Translator](https://github.com/Azure/co-op-translator). Несмотря на наши усилия по обеспечению точности, имейте в виду, что автоматический перевод может содержать ошибки или неточности. Оригинальный документ на его исходном языке следует считать авторитетным источником. Для получения критически важной информации рекомендуется обратиться к профессиональному человеческому переводу. Мы не несем ответственности за любые недоразумения или неправильные толкования, возникшие в результате использования этого перевода.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
